### Requirements setup

In [ ]:
pip install ftfy regex tqdm

In [ ]:
pip install git+https://github.com/openai/CLIP.git

In [ ]:
pip install matplotlib seaborn scikit-learn

In [ ]:
import os
import random
import copy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models, datasets
from PIL import Image

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

### Data preprocessing

In [ ]:
DATA_ROOT = './OfficeHomeDataset'
ART_DIR = os.path.join(DATA_ROOT, 'Art')
REAL_DIR = os.path.join(DATA_ROOT, 'Real World')

assert os.path.isdir(ART_DIR), f'Missing source dir: {ART_DIR}'
assert os.path.isdir(REAL_DIR), f'Missing target dir: {REAL_DIR}'

CLASSES = sorted(os.listdir(ART_DIR))
CLASSES = [c for c in CLASSES if os.path.isdir(os.path.join(ART_DIR, c))]
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
print(f'Number of classes: {NUM_CLASSES}')
print(f'First 10 classes: {CLASSES[:10]}')

In [ ]:
class OfficeHomeDataset(Dataset):
    def __init__(self, root, classes, transform=None):
        self.transform = transform
        self.samples = []
        self.classes = classes
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        for c in classes:
            cdir = os.path.join(root, c)
            if not os.path.isdir(cdir):
                continue
            for fname in os.listdir(cdir):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                    self.samples.append((os.path.join(cdir, fname),
                                         self.class_to_idx[c]))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)
        return img, label, idx


class PseudoLabeledDataset(Dataset):
    def __init__(self, base_ds, pseudo_labels, selected_mask, transform=None):
        self.base_ds = base_ds
        self.pseudo_labels = pseudo_labels
        self.selected_mask = selected_mask

        self.selected_indices = np.where(selected_mask)[0]
        self.transform = transform
    def __len__(self):
        return len(self.selected_indices)
    def __getitem__(self, i):
        orig_idx = int(self.selected_indices[i])
        path, _ = self.base_ds.samples[orig_idx]
        img = Image.open(path).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)
        plabel = int(self.pseudo_labels[orig_idx])
        return img, plabel

In [ ]:
# image net transformation
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

source_train = OfficeHomeDataset(ART_DIR,  CLASSES, transform=train_tf)
source_eval  = OfficeHomeDataset(ART_DIR,  CLASSES, transform=eval_tf)
target_train = OfficeHomeDataset(REAL_DIR, CLASSES, transform=train_tf)
target_eval  = OfficeHomeDataset(REAL_DIR, CLASSES, transform=eval_tf)

print(f'Source (Art)        : {len(source_train)} images')
print(f'Target (Real World) : {len(target_train)} images')

BATCH = 64
NUM_WORKERS = 4

source_train_loader = DataLoader(source_train, batch_size=BATCH, shuffle=True,
                                  num_workers=NUM_WORKERS, drop_last=True)
source_eval_loader  = DataLoader(source_eval,  batch_size=BATCH, shuffle=False,
                                  num_workers=NUM_WORKERS)
target_eval_loader  = DataLoader(target_eval,  batch_size=BATCH, shuffle=False,
                                  num_workers=NUM_WORKERS)

### Task 1 - Output Space UDA

In [ ]:
def build_model(num_classes=NUM_CLASSES):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    in_feat = model.fc.in_features
    model.fc = nn.Linear(in_feat, num_classes)
    return model

def make_optimizer(model, lr_fc=1e-3, lr_backbone=1e-4, wd=5e-4):
    fc_params = list(model.fc.parameters())
    fc_ids = set(id(p) for p in fc_params)
    backbone_params = [p for p in model.parameters() if id(p) not in fc_ids]
    return torch.optim.SGD([
        {'params': backbone_params, 'lr': lr_backbone},
        {'params': fc_params,       'lr': lr_fc},
    ], momentum=0.9, weight_decay=wd, nesterov=True)

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion,
                    desc='train', ignore_index=-1):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    pbar = tqdm(loader, desc=desc, leave=False)
    for batch in pbar:
        if len(batch) == 3:
            x, y, _ = batch
        else:
            x, y = batch
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            mask = (y != ignore_index)
            pred = logits.argmax(1)
            correct += ((pred == y) & mask).sum().item()
            total += mask.sum().item()
            loss_sum += loss.item() * x.size(0)
    return loss_sum / max(1, len(loader.dataset)), correct / max(1, total)

@torch.no_grad()
def evaluate(model, loader, return_preds=False):
    model.eval()
    all_pred, all_true, all_prob = [], [], []
    for batch in loader:
        if len(batch) == 3:
            x, y, _ = batch
        else:
            x, y = batch
        x = x.to(device)
        logits = model(x)
        prob = F.softmax(logits, 1).cpu().numpy()
        pred = prob.argmax(1)
        all_pred.append(pred)
        all_true.append(y.numpy())
        all_prob.append(prob)
    all_pred = np.concatenate(all_pred)
    all_true = np.concatenate(all_true)
    all_prob = np.concatenate(all_prob)
    acc = (all_pred == all_true).mean()

    per_class = []
    for c in range(NUM_CLASSES):
        m = all_true == c
        if m.sum() > 0:
            per_class.append((all_pred[m] == c).mean())
    mca = float(np.mean(per_class))
    if return_preds:
        return acc, mca, all_pred, all_true, all_prob
    return acc, mca

### Baseline source-only model training

In [ ]:
SRC_EPOCHS = 40

model_src = build_model().to(device)
optimizer = make_optimizer(model_src)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SRC_EPOCHS)
criterion = nn.CrossEntropyLoss()

history_src = {'src_acc': [], 'tgt_acc': []}
for ep in range(SRC_EPOCHS):
    tl, ta = train_one_epoch(model_src, source_train_loader, optimizer, criterion,
                              desc=f'Src ep{ep+1}')
    scheduler.step()
    sa, _ = evaluate(model_src, source_eval_loader)
    ta_t, mca_t = evaluate(model_src, target_eval_loader)
    history_src['src_acc'].append(sa)
    history_src['tgt_acc'].append(ta_t)
    print(f'Ep {ep+1:02d}  src_acc={sa:.4f}  tgt_acc={ta_t:.4f}  tgt_mca={mca_t:.4f}')

src_only_tgt_acc, src_only_tgt_mca = evaluate(model_src, target_eval_loader)
print(f'\n[Source-Only] Target Acc = {src_only_tgt_acc:.4f} | Mean-Class Acc = {src_only_tgt_mca:.4f}')

#saving model
torch.save(model_src.state_dict(), 'model_src_only.pt')

In [ ]:
@torch.no_grad()
def predict_target_probs(model, base_dataset, batch=BATCH):
    loader = DataLoader(base_dataset, batch_size=batch, shuffle=False,
                         num_workers=NUM_WORKERS)
    model.eval()
    probs = np.zeros((len(base_dataset), NUM_CLASSES), dtype=np.float32)
    offset = 0
    for x, _, _ in loader:
        x = x.to(device)
        p = F.softmax(model(x), 1).cpu().numpy()
        probs[offset:offset + p.shape[0]] = p
        offset += p.shape[0]
    preds = probs.argmax(1)
    return probs, preds


def select_vanilla(probs, preds, portion=0.3):
    conf = probs.max(1)
    k = int(portion * len(conf))
    if k <= 0:
        return np.zeros_like(conf, dtype=bool)
    thresh = np.partition(conf, -k)[-k]
    return conf >= thresh


def select_class_balanced(probs, preds, portion=0.3):
    mask = np.zeros(len(preds), dtype=bool)
    conf = probs.max(1)
    for c in range(NUM_CLASSES):
        idxs = np.where(preds == c)[0]
        if len(idxs) == 0:
            continue
        k = max(1, int(portion * len(idxs)))
        k = min(k, len(idxs))
        top_k = idxs[np.argpartition(-conf[idxs], k - 1)[:k]]
        mask[top_k] = True
    return mask


def select_crst(probs, preds, portion=0.3):
    return select_class_balanced(probs, preds, portion)


def pseudo_label_stats(mask, preds, true_labels):
    n_sel = int(mask.sum())
    if n_sel == 0:
        return {'selected': 0, 'pseudo_acc': 0.0, 'per_class': Counter()}
    correct = (preds[mask] == true_labels[mask]).mean()
    per_cls = Counter(preds[mask].tolist())
    return {'selected': n_sel, 'pseudo_acc': float(correct), 'per_class': per_cls}

In [ ]:
class CRSTLoss(nn.Module):
    def __init__(self, lam=0.1):
        super().__init__()
        self.lam = lam
        self.ce = nn.CrossEntropyLoss()
    def forward(self, logits, target):
        ce = self.ce(logits, target)
        logp = F.log_softmax(logits, dim=1)
        p = logp.exp()
        ent = -(p * logp).sum(1).mean()
        # Minimise CE while maximising entropy (regularisation)
        return ce - self.lam * ent


def self_training_run(method='st',
                      n_rounds=3,
                      epochs_per_round=3,
                      portion_schedule=(0.2, 0.35, 0.5),
                      crst_lambda=0.1,
                      init_ckpt='model_src_only.pt'):

    assert method in ('st', 'cbst', 'crst')
    model = build_model().to(device)
    model.load_state_dict(torch.load(init_ckpt, map_location=device))

    if method == 'crst':
        criterion = CRSTLoss(lam=crst_lambda)
        select_fn = select_crst
    elif method == 'cbst':
        criterion = nn.CrossEntropyLoss()
        select_fn = select_class_balanced
    else:
        criterion = nn.CrossEntropyLoss()
        select_fn = select_vanilla

    history = {'round': [], 'portion': [], 'pseudo_acc': [],
               'selected': [], 'tgt_acc': [], 'tgt_mca': []}

    true_tgt_labels = np.array([lbl for _, lbl in target_eval.samples])

    for r in range(n_rounds):
        portion = portion_schedule[min(r, len(portion_schedule) - 1)]

        #generating pseudo labels
        probs, preds = predict_target_probs(model, target_eval)
        mask = select_fn(probs, preds, portion=portion)
        stats = pseudo_label_stats(mask, preds, true_tgt_labels)
        print(f'\n[{method.upper()}] Round {r+1}: portion={portion:.2f} '
              f'selected={stats["selected"]} pseudo_acc={stats["pseudo_acc"]:.4f}')

        #wrapping targets with the pseudo
        pseudo_ds = PseudoLabeledDataset(target_eval, preds, mask, transform=train_tf)
        pseudo_loader = DataLoader(pseudo_ds, batch_size=BATCH, shuffle=True,
                                     num_workers=NUM_WORKERS, drop_last=True)

        #training
        optimizer = make_optimizer(model, lr_fc=5e-4, lr_backbone=5e-5)
        for ep in range(epochs_per_round):
            model.train()
            src_iter = iter(source_train_loader)
            pbar = tqdm(pseudo_loader, desc=f'{method}-r{r+1}-ep{ep+1}', leave=False)
            for xt, yt in pbar:
                try:
                    xs, ys, _ = next(src_iter)
                except StopIteration:
                    src_iter = iter(source_train_loader)
                    xs, ys, _ = next(src_iter)
                xs, ys = xs.to(device), ys.to(device)
                xt, yt = xt.to(device), yt.to(device)
                logits_s = model(xs)
                logits_t = model(xt)
                loss = nn.functional.cross_entropy(logits_s, ys) + criterion(logits_t, yt)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        #evalluation
        ta, mca = evaluate(model, target_eval_loader)
        history['round'].append(r + 1)
        history['portion'].append(portion)
        history['pseudo_acc'].append(stats['pseudo_acc'])
        history['selected'].append(stats['selected'])
        history['tgt_acc'].append(ta)
        history['tgt_mca'].append(mca)
        print(f'  → tgt_acc={ta:.4f}  tgt_mca={mca:.4f}')

    return model, history

In [ ]:
#runnnig training for all methods
model_st,   hist_st   = self_training_run(method='st')
model_cbst, hist_cbst = self_training_run(method='cbst')
model_crst, hist_crst = self_training_run(method='crst', crst_lambda=0.1)

### Task 1 -- Results visualization and analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

#per-round target accuracy
ax = axes[0]
ax.axhline(src_only_tgt_acc, ls='--', c='gray', label=f'Source-Only ({src_only_tgt_acc:.3f})', lw=2)
for name, h, color in [('ST', hist_st, 'tab:red'),
                       ('CBST', hist_cbst, 'tab:green'),
                       ('CRST', hist_crst, 'tab:blue')]:
    rounds = [0] + h['round']
    accs = [src_only_tgt_acc] + h['tgt_acc']
    ax.plot(rounds, accs, marker='o', lw=2, label=name, color=color)
ax.set_xlabel('Self-training round'); ax.set_ylabel('Target Accuracy')
ax.set_title('Target Accuracy Across Rounds'); ax.legend(); ax.grid(alpha=.3)

#per-round pseudo label accuracy
ax = axes[1]
for name, h, color in [('ST', hist_st, 'tab:red'),
                       ('CBST', hist_cbst, 'tab:green'),
                       ('CRST', hist_crst, 'tab:blue')]:
    ax.plot(h['round'], h['pseudo_acc'], marker='s', lw=2, label=name, color=color)
ax.set_xlabel('Self-training round'); ax.set_ylabel('Pseudo-label accuracy')
ax.set_title('Pseudo-Label Quality (on selected samples)')
ax.legend(); ax.grid(alpha=.3)

plt.tight_layout()
plt.show()

In [ ]:
#generating class distributions
@torch.no_grad()
def get_selection_class_distribution(model, select_fn, portion):
    probs, preds = predict_target_probs(model, target_eval)
    mask = select_fn(probs, preds, portion=portion)
    selected_preds = preds[mask]
    counts = np.bincount(selected_preds, minlength=NUM_CLASSES)
    return counts

probs_src, preds_src = predict_target_probs(model_src, target_eval)
mask_st   = select_vanilla(probs_src, preds_src, portion=0.5)
mask_cbst = select_class_balanced(probs_src, preds_src, portion=0.5)

dist_st   = np.bincount(preds_src[mask_st],   minlength=NUM_CLASSES)
dist_cbst = np.bincount(preds_src[mask_cbst], minlength=NUM_CLASSES)

fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)
x = np.arange(NUM_CLASSES)
axes[0].bar(x, dist_st, color='tab:red')
axes[0].set_title(f'Vanilla ST: per-class pseudo-label counts (portion=0.5)  — '
                   f'{(dist_st == 0).sum()} classes get ZERO samples')
axes[0].set_ylabel('# selected')
axes[0].grid(alpha=.3, axis='y')

axes[1].bar(x, dist_cbst, color='tab:green')
axes[1].set_title(f'CBST: per-class pseudo-label counts (portion=0.5)  — '
                   f'{(dist_cbst == 0).sum()} classes get ZERO samples')
axes[1].set_ylabel('# selected'); axes[1].set_xlabel('Class index')
axes[1].grid(alpha=.3, axis='y')

plt.xticks(x, CLASSES, rotation=90, fontsize=6)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import *

# Collect predictions from each model (reuse if already computed)
_, _, pred_src,  true_t, prob_src  = evaluate(model_src,  target_eval_loader, return_preds=True)
_, _, pred_st,   _,      prob_st   = evaluate(model_st,   target_eval_loader, return_preds=True)
_, _, pred_cbst, _,      prob_cbst = evaluate(model_cbst, target_eval_loader, return_preds=True)
_, _, pred_crst, _,      prob_crst = evaluate(model_crst, target_eval_loader, return_preds=True)


def compute_all_metrics(y_true, y_pred, num_classes=NUM_CLASSES):
    #calculating metrics

    #accuracy
    acc = accuracy_score(y_true, y_pred)

    #mean class accuracy
    per_class_acc = []
    for c in range(num_classes):
        m = y_true == c
        if m.sum() > 0:
            per_class_acc.append((y_pred[m] == c).mean())
    mca = float(np.mean(per_class_acc))

    #macro and weighted F1 scores
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0,
                         labels=list(range(num_classes)))
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0,
                            labels=list(range(num_classes)))

    return {
        'Accuracy':     acc,
        'MCA':          mca,
        'Macro-F1':     macro_f1,
        'Weighted-F1':  weighted_f1
    }

rows = []
for name, pred in [('Source-Only', pred_src),
                    ('Vanilla ST', pred_st),
                    ('CBST', pred_cbst),
                    ('CRST', pred_crst)]:
    m = compute_all_metrics(true_t, pred)
    m = {'Method': name, **m}
    rows.append(m)

metrics_df = pd.DataFrame(rows)
metrics_df

In [ ]:
def denormalize(t):
    m = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    s = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (t * s + m).clamp(0, 1)

def show_rectifications(indices, title, n=8, use_confidence=True):
    if len(indices) == 0:
        print(f'No examples for: {title}'); return

    if use_confidence:
        st_conf_on_wrong = prob_st[indices].max(axis=1)
        order = np.argsort(-st_conf_on_wrong)
        sel = indices[order[:n]]
    else:
        sel = np.random.RandomState(SEED).choice(indices, min(n, len(indices)), replace=False)

    n = len(sel)
    cols = 4
    rows = int(np.ceil(n / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 5.6 * rows))
    axes = np.array(axes).flatten()

    for ax, idx in zip(axes, sel):
        img, _, _ = target_eval[int(idx)]
        ax.imshow(denormalize(img).permute(1, 2, 0).numpy())
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

        true_c = CLASSES[true_t[idx]]
        st_c, cbst_c, crst_c = CLASSES[pred_st[idx]], CLASSES[pred_cbst[idx]], CLASSES[pred_crst[idx]]

        st_conf   = prob_st[idx,   pred_st[idx]]
        cbst_conf = prob_cbst[idx, pred_cbst[idx]]
        crst_conf = prob_crst[idx, pred_crst[idx]]

        cbst_color = 'green' if pred_cbst[idx] == true_t[idx] else 'red'
        crst_color = 'green' if pred_crst[idx] == true_t[idx] else 'red'

        #showing ground truth
        ax.set_title(f'True: {true_c}', fontsize=11, color='black', pad=8)
        #showing predictions of all methods
        ax.text(0.5, -0.06, f'ST:   {st_c} ({st_conf:.2f})',
                 ha='center', transform=ax.transAxes, color='red',      fontsize=10)
        ax.text(0.5, -0.14, f'CBST: {cbst_c} ({cbst_conf:.2f})',
                 ha='center', transform=ax.transAxes, color=cbst_color, fontsize=10)
        ax.text(0.5, -0.22, f'CRST: {crst_c} ({crst_conf:.2f})',
                 ha='center', transform=ax.transAxes, color=crst_color, fontsize=10)



    fig.suptitle(title, fontsize=14, y=1.00)
    plt.subplots_adjust(hspace=0.6, wspace=0.25)
    plt.tight_layout()
    safe = title.split(':')[0].strip().replace(' ', '_').replace('/', '_')
    plt.savefig(f'task1_rectify_{safe}.png', dpi=120, bbox_inches='tight')
    plt.show()


show_rectifications(both_fix,
                     'BOTH CBST & CRST rectify: ST was confidently wrong',
                     n=8, use_confidence=True)

### Task 2 -- LMM Knowldge Transfer

In [ ]:
import clip

CLIP_MODEL_NAME = 'ViT-B/16'
clip_model, clip_preprocess = clip.load(CLIP_MODEL_NAME, device=device)
clip_model.eval()
for p in clip_model.parameters():
    p.requires_grad = False

CLIP_FEAT_DIM = clip_model.visual.output_dim
print(f'CLIP model: {CLIP_MODEL_NAME}, feature dim: {CLIP_FEAT_DIM}')

In [ ]:
def classname_to_text(cn):
    return cn.replace('_', ' ').lower().strip()

#prompt ensembling
PROMPT_TEMPLATES = [
    'a photo of a {}.',
    'a picture of a {}.',
    'an image of a {}.',
    'a rendering of a {}.',
    'a close-up photo of a {}.',
]

@torch.no_grad()
def compute_text_classifier(classnames, templates=PROMPT_TEMPLATES):
    weights = []
    for cn in classnames:
        texts = [t.format(classname_to_text(cn)) for t in templates]
        tokens = clip.tokenize(texts).to(device)
        emb = clip_model.encode_text(tokens).float()
        emb = emb / emb.norm(dim=-1, keepdim=True)
        emb = emb.mean(0)
        emb = emb / emb.norm()
        weights.append(emb)
    return torch.stack(weights, dim=1).to(device)  # [D, C]

text_weights = compute_text_classifier(CLASSES)
print('Text classifier shape:', text_weights.shape)

In [ ]:
source_clip = OfficeHomeDataset(ART_DIR,  CLASSES, transform=clip_preprocess)
target_clip = OfficeHomeDataset(REAL_DIR, CLASSES, transform=clip_preprocess)

@torch.no_grad()
def extract_clip_features(dataset, batch=128):
    loader = DataLoader(dataset, batch_size=batch, shuffle=False,
                          num_workers=NUM_WORKERS)
    feats, labels = [], []
    for x, y, _ in tqdm(loader, desc='CLIP features'):
        x = x.to(device)
        f = clip_model.encode_image(x).float()
        f = f / f.norm(dim=-1, keepdim=True)
        feats.append(f.cpu()); labels.append(y)
    return torch.cat(feats), torch.cat(labels)

F_tgt, Y_tgt = extract_clip_features(target_clip)
F_src, Y_src = extract_clip_features(source_clip)

### Zero-shot CLIP

In [ ]:
def zero_shot_accuracy(feats, labels, text_w):
    logits = 100.0 * feats.to(device) @ text_w   # [N, C]
    pred = logits.argmax(1).cpu()
    acc = (pred == labels).float().mean().item()
    per_class = []
    for c in range(NUM_CLASSES):
        m = labels == c
        if m.sum() > 0:
            per_class.append((pred[m] == c).float().mean().item())
    return acc, float(np.mean(per_class)), pred.numpy()

zs_acc, zs_mca, zs_pred = zero_shot_accuracy(F_tgt, Y_tgt, text_weights)
zs_Acc

### creatig examples that adapters would see

In [ ]:
def build_few_shot(F_all, Y_all, k_shot=16, seed=SEED):
    rng = np.random.RandomState(seed)
    keys, lbls = [], []
    for c in range(NUM_CLASSES):
        idx = np.where(Y_all.numpy() == c)[0]
        if len(idx) == 0:
            continue
        pick = rng.choice(idx, size=min(k_shot, len(idx)), replace=False)
        keys.append(F_all[pick]); lbls.append(torch.full((len(pick),), c))
    return torch.cat(keys), torch.cat(lbls)

K = 16
F_fs, Y_fs = build_few_shot(F_src, Y_src, k_shot=K)
print(f'Few-shot support: {F_fs.shape[0]} images ({K} per class)')

### CLIP-Adapter

In [ ]:
class ClipAdapter(nn.Module):
    def __init__(self, dim, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(dim, dim // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(dim // reduction, dim, bias=False),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.fc(x)

def train_clip_adapter(F_fs, Y_fs, text_w, alpha=0.2, epochs=200, lr=1e-3):
    adapter = ClipAdapter(CLIP_FEAT_DIM).to(device)
    optim = torch.optim.AdamW(adapter.parameters(), lr=lr, eps=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    F_fs_d = F_fs.to(device); Y_fs_d = Y_fs.to(device)

    for ep in range(epochs):
        adapter.train()
        idx = torch.randperm(F_fs_d.size(0))
        total_loss = 0.0
        for i in range(0, len(idx), 256):
            b = idx[i:i+256]
            f = F_fs_d[b]; y = Y_fs_d[b]
            a = adapter(f)
            f_new = alpha * a + (1 - alpha) * f
            f_new = f_new / f_new.norm(dim=-1, keepdim=True)
            logits = 100.0 * f_new @ text_w
            loss = F.cross_entropy(logits, y)
            optim.zero_grad(); loss.backward(); optim.step()
            total_loss += loss.item()
        sched.step()
    return adapter

@torch.no_grad()
def eval_clip_adapter(adapter, F, Y, text_w, alpha=0.2):
    adapter.eval()
    f = F.to(device)
    a = adapter(f)
    f_new = alpha * a + (1 - alpha) * f
    f_new = f_new / f_new.norm(dim=-1, keepdim=True)
    logits = 100.0 * f_new @ text_w
    pred = logits.argmax(1).cpu()
    acc = (pred == Y).float().mean().item()
    per_class = []
    for c in range(NUM_CLASSES):
        m = Y == c
        if m.sum() > 0:
            per_class.append((pred[m] == c).float().mean().item())
    return acc, float(np.mean(per_class)), pred.numpy()

ALPHA_CA = 0.2
adapter = train_clip_adapter(F_fs, Y_fs, text_weights, alpha=ALPHA_CA, epochs=200, lr=1e-3)
ca_acc, ca_mca, ca_pred = eval_clip_adapter(adapter, F_tgt, Y_tgt, text_weights, alpha=ALPHA_CA)
ca_acc

### Tip-Adapter

In [ ]:
def build_tip_cache(F_fs, Y_fs, num_classes=NUM_CLASSES):
    K_cache = F_fs.to(device)
    L = F.one_hot(Y_fs, num_classes).float().to(device)
    return K_cache, L

@torch.no_grad()
def tip_logits(features, K_cache, L, text_w, alpha, beta):
    f = features.to(device)
    sims = f @ K_cache.t()
    phi  = torch.exp(-beta * (1 - sims))
    cache_logits = phi @ L
    clip_logits  = 100.0 * f @ text_w
    return clip_logits + alpha * cache_logits

def eval_tip(features, labels, K_cache, L, text_w, alpha, beta):
    logits = tip_logits(features, K_cache, L, text_w, alpha, beta)
    pred = logits.argmax(1).cpu()
    acc = (pred == labels).float().mean().item()
    per_class = []
    for c in range(NUM_CLASSES):
        m = labels == c
        if m.sum() > 0:
            per_class.append((pred[m] == c).float().mean().item())
    return acc, float(np.mean(per_class)), pred.numpy()

K_cache, L_cache = build_tip_cache(F_fs, Y_fs)

best = (None, None, -1)
for a in [0.5, 1.0, 2.0, 3.0, 5.0]:
    for b in [1.0, 2.5, 5.0, 5.5]:
        acc, _, _ = eval_tip(F_tgt, Y_tgt, K_cache, L_cache, text_weights, a, b)
        if acc > best[2]:
            best = (a, b, acc)
ALPHA_TIP, BETA_TIP, _ = best
print(f'Tip-Adapter best (alpha, beta) = ({ALPHA_TIP}, {BETA_TIP})')
tip_acc, tip_mca, tip_pred = eval_tip(F_tgt, Y_tgt, K_cache, L_cache,
                                        text_weights, ALPHA_TIP, BETA_TIP)
tip_acc

### Tip-Adapter-F

In [ ]:
def train_tip_f(F_fs, Y_fs, text_w, alpha=1.0, beta=5.5, epochs=200, lr=1e-3):
    K_param = nn.Parameter(F_fs.to(device).clone())      # [NK, D]
    L = F.one_hot(Y_fs, NUM_CLASSES).float().to(device)
    optim = torch.optim.AdamW([K_param], lr=lr, eps=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    F_fs_d = F_fs.to(device); Y_fs_d = Y_fs.to(device)
    for ep in range(epochs):
        idx = torch.randperm(F_fs_d.size(0))
        for i in range(0, len(idx), 256):
            b_ = idx[i:i+256]
            f = F_fs_d[b_]; y = Y_fs_d[b_]
            sims = f @ K_param.t()
            phi  = torch.exp(-beta * (1 - sims))
            cache_logits = phi @ L
            clip_logits  = 100.0 * f @ text_w
            logits = clip_logits + alpha * cache_logits
            loss = F.cross_entropy(logits, y)
            optim.zero_grad(); loss.backward(); optim.step()
        sched.step()
    return K_param.detach(), L

K_f, L_f = train_tip_f(F_fs, Y_fs, text_weights,
                        alpha=ALPHA_TIP, beta=BETA_TIP, epochs=200)
tipf_acc, tipf_mca, tipf_pred = eval_tip(F_tgt, Y_tgt, K_f, L_f,
                                           text_weights, ALPHA_TIP, BETA_TIP)
tipf_acc

### Task 2 -- Results visualization and analysis

In [ ]:
y_true_np = Y_tgt.numpy()
conf_zs   = probs_zs  [np.arange(len(y_true_np)), pred_zs_c]
conf_ca   = probs_ca  [np.arange(len(y_true_np)), pred_ca_c]
conf_tip  = probs_tip [np.arange(len(y_true_np)), pred_tip_c]
conf_tipf = probs_tipf[np.arange(len(y_true_np)), pred_tipf_c]


def denormalize_clip(t):
    m = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(3, 1, 1)
    s = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3, 1, 1)
    return (t * s + m).clamp(0, 1)


def show_predictions_grid(indices, title, n=8):
    if len(indices) == 0:
        print(f'No examples for: {title}'); return
    n = min(n, len(indices))
    sel = np.random.RandomState(SEED).choice(indices, n, replace=False)

    cols = 4
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 6.0 * rows))
    axes = np.array(axes).flatten()

    for ax, idx in zip(axes, sel):
        img, _, _ = target_clip[int(idx)]
        ax.imshow(denormalize_clip(img).permute(1, 2, 0).numpy())
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

        true_c = CLASSES[y_true_np[idx]]
        rows_data = [
            ('ZS',   pred_zs_c  [idx], conf_zs  [idx]),
            ('CA',   pred_ca_c  [idx], conf_ca  [idx]),
            ('Tip',  pred_tip_c [idx], conf_tip [idx]),
            ('TipF', pred_tipf_c[idx], conf_tipf[idx]),
        ]

        ax.set_title(f'True: {true_c}', fontsize=11, color='black', pad=8)
        for i, (name, pred_c, conf) in enumerate(rows_data):
            correct = pred_c == y_true_np[idx]
            colour  = 'green' if correct else 'red'
            y_pos   = -0.06 - i * 0.08
            ax.text(0.5, y_pos, f'{name:5s}: {CLASSES[pred_c]} ({conf:.2f})',
                     ha='center', transform=ax.transAxes, color=colour, fontsize=10)

    fig.suptitle(title, fontsize=14, y=1.00)
    plt.subplots_adjust(hspace=0.8, wspace=0.25)
    plt.tight_layout()
    safe = title.split(':')[0].strip().replace(' ', '_').replace('/', '_')
    plt.savefig(f'task2_qual_{safe}.png', dpi=120, bbox_inches='tight')
    plt.show()

zs_wrong     = pred_zs_c   != y_true_np
ca_right     = pred_ca_c   == y_true_np
tip_right    = pred_tip_c  == y_true_np
tipf_right   = pred_tipf_c == y_true_np

all_correct = np.where(~zs_wrong & ca_right & tip_right & tipf_right)[0]

adapters_rescue = np.where(zs_wrong & ca_right & tip_right & tipf_right)[0]

only_tipf_rescues = np.where(zs_wrong & ~ca_right & ~tip_right & tipf_right)[0]

all_fail = np.where(zs_wrong & ~ca_right & ~tip_right & ~tipf_right)[0]


show_predictions_grid(adapters_rescue,
                       'Few-shot adapters RESCUE Zero-Shot (all 3 adapters correct)',
                       n=8)